### ✨ Train Your Own SMILES Classifier!

Welcome to the SMILES Classifier training lab! This is a very lightweight model—even on Colab's free GPU, training will be complete in just a few minutes.

💡 **Pro Tip: Activate the GPU for a Lightning-Fast Experience!**

For blazing-fast training speeds, we highly recommend enabling the free T4 GPU.
Just click on **Runtime** → **Change runtime type** at the top-right of the page, and select **T4 GPU** from the *Hardware accelerator* dropdown.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Your 3-Step Launch Sequence (Crucial!)

1.  **Activate GPU & Connect**: First, please **enable the T4 GPU** as advised above. Then, look to the top-right corner. If it's not already showing "Connecting...", please click the **Connect** button manually.

2.  **Wait for the Green Checkmark ✓**: Wait a moment until a green checkmark appears next to your `RAM` and `Disk` status. This confirms you're connected to a powerful GPU and ready to go!

3.  **Run All Cells**: Now, simply go to the **Runtime** menu at the top and click **Run all**. It's the perfect time to grab a quick cup of coffee!

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 📈 Once Training Begins, You'll See...

Our automated pipeline will kick off, allowing you to clearly monitor the model's learning process:

*   **Live Progress Bars**: Visually track the model's progress epoch by epoch.
*   **Performance Metrics**: After each epoch, the training and validation loss will be printed, giving you insight into how well the model is learning.
*   **Best Model Checkpoints**: The system automatically saves the best-performing model based on validation results, no manual intervention needed.
*   **Your Custom Classifier**: In just a few minutes, your freshly trained classifier model will be ready and saved to your Google Drive!

---

In [ ]:
# @title 1. Project Setup & All Necessary Downloads
# @markdown This notebook clones the `PS-Classifier` project, installs dependencies, and downloads all required pre-trained model weights from Hugging Face for both `CSU-IR` and `PS-Classifier`.
# @markdown Run this cell once to set up your entire environment.

# ==============================================================================
#  PS-Classifier & CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted.")


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================
print("\n--- Defining Project Paths ---")
# The parent folder on your Drive where repositories are stored
GDRIVE_PARENT_PATH = "/content/drive/MyDrive/colab_repos"
# The root path for the PS-Classifier project
PROJECT_ROOT = os.path.join(GDRIVE_PARENT_PATH, "PS-Classifier")

# Requirements file path
REQUIREMENTS_FILE = os.path.join(PROJECT_ROOT, 'requirements_colab.txt')

# --- Destination Folders for Weights on Google Drive ---
# This is where the downloaded SMILES encoder from CSU-IR will be saved.
DEST_CSU_IR_WEIGHTS_PATH = os.path.join(PROJECT_ROOT, 'check_points')


print(f"✅ Project Root set to: {PROJECT_ROOT}")
print(f"✅ CSU-IR weights will be saved to: {DEST_CSU_IR_WEIGHTS_PATH}")



# ==================================================================
# --- 3. Clone or Update the 'PS-Classifier' repository ---
# ==================================================================
print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
    # Configure git to use the token for authentication
    !git config --global url."https://{GIT_TOKEN}:@github.com/".insteadOf "https://github.com/"
    print("✅ GitHub token configured.")
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your GitHub Personal Access Token in Colab Secrets (secret name: GITHUB_PAT)")

if not os.path.exists(PROJECT_ROOT):
    print(f"⏳ Cloning 'PS-Classifier' to: {PROJECT_ROOT}")
    !git clone -q https://github.com/Hsqcsu/PS-Classifier.git {PROJECT_ROOT}
else:
    print(f"✅ Repository already exists. Pulling latest changes...")
    # Use -C flag to run git commands in the specified directory
    !git -C {PROJECT_ROOT} pull


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ Hugging Face token found.")
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                # Use move instead of copy for efficiency with large files
                shutil.move(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully to {target_path}")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"

# 1. Weights for the pre-trained SMILES encoder (from CSU-IR)
CSU_IR_SMILES_ENCODER_TO_DOWNLOAD = {
    "Pretrain_model_weight/best_smiles_model_650-4000.pth": "best_smiles_model_650-4000.pth"
}


# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")

# Download the required CSU-IR pre-trained model
download_files_from_hf(HF_REPO_ID, CSU_IR_SMILES_ENCODER_TO_DOWNLOAD, DEST_CSU_IR_WEIGHTS_PATH, HF_TOKEN)


# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")
print("You can now open and run the 'Train_SMILES_Classifier.ipynb' notebook.")

---
### ** Environment & Model Initialization** 🧠

*Load the Engine Room*

This is the most critical setup cell. It performs all the heavy lifting to prepare our environment by:
1.  **Setting up all necessary paths** to your data and code within Google Drive.
2.  **Importing all custom Python modules**, including the `SMILES_encoder` and `Classifier_model` architectures.
3.  **Initializing the models** on the CPU and **freezing the pre-trained encoder**, getting them ready for fine-tuning.

**Please run this cell once.** After it completes, all the foundational components will be loaded and ready to go.

In [ ]:
# @title 1. Environment & Model Initialization
# @markdown This cell sets up paths, imports all necessary modules from the cloned repository, and initializes the models.

import sys
import os
import yaml
import json
import argparse
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from tqdm.notebook import tqdm # Use tqdm.notebook for better UI in Colab

# --- 1. Project Root Setup ---
# This assumes the Setup notebook has already run and cloned the repo.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    GDRIVE_PARENT_PATH = "/content/drive/MyDrive/colab_repos"
    PROJECT_ROOT = os.path.join(GDRIVE_PARENT_PATH, "PS-Classifier")
    print(f"✅ Project Root set to: {PROJECT_ROOT}")
except ImportError:
    PROJECT_ROOT = "path/to/your/local/PS-Classifier" # For local testing
    print(f"⚠️ Not in Colab. Project Root set to: {PROJECT_ROOT}")

# Add project to Python path to allow imports
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom modules
try:
    from model.SMILES_encoder import SmilesModel
    from model.Classifier_model import classifymodel
    print("✅ Custom models imported successfully.")
except (ModuleNotFoundError, ImportError) as e:
    print(f"❌ Module import failed! Error: {e}")
    print(f"Please ensure the repository was cloned correctly at: {PROJECT_ROOT}")
    raise e

# --- 2. Load Configuration from YAML file ---
# Since this is a self-contained notebook, we define the config here.
# In a real-world scenario, this could also be loaded from a file.
CONFIG = {
    'paths': {
        'output_dir': os.path.join(PROJECT_ROOT,"check_points/colab_trained"),
        'tokenizer': os.path.join(PROJECT_ROOT, "model/tokenizer-smiles-roberta-1e_new"),
        'pretrained_smiles_encoder': os.path.join(PROJECT_ROOT, "check_points/best_smiles_model_650-4000.pth"),
        'train_smiles': os.path.join(PROJECT_ROOT, "data/pretrain_data/SMILES_Classifier/train_smiles.txt"),
        'train_labels': os.path.join(PROJECT_ROOT, "data/pretrain_data/SMILES_Classifier/train_labels.pt"),
        'val_smiles': os.path.join(PROJECT_ROOT, "data/pretrain_data/SMILES_Classifier/val_smiles.txt"),
        'val_labels': os.path.join(PROJECT_ROOT, "data/pretrain_data/SMILES_Classifier/val_labels.pt"),
    },
    'model_params': {
        'smiles_encoder': {'smiles_maxlen': 300, 'max_position_embeddings': 505, 'vocab_size': 181, 'feature_dim': 768},
        'classifier': {'dim': 1024, 'num_classes': 2},
    },
    'training_params': {
        'num_epochs': 100,
        'device': 'auto'
    },
    'dataloader_params': {
        'batch_size': 208,
        'num_workers': 2 # Good starting point for Colab
    },
    'optimizer_params': {
        'learning_rate': 5.0e-5,
        'weight_decay': 1.0e-4
    },
    'model_save_params': {
        'num_best_models': 5,
        'loss_log_file': "loss_data.json"
    }
}
print("✅ Configuration loaded.")

# --- 3. Initialize Models ---
print("Initializing models...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize and setup SMILES Encoder (Feature Extractor)
smiles_encoder_config = CONFIG['model_params']['smiles_encoder']
smiles_encoder = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=CONFIG['paths']['tokenizer'], **smiles_encoder_config)
print(f"Loading pre-trained SMILES encoder from: {CONFIG['paths']['pretrained_smiles_encoder']}")
smiles_encoder.load_weights(CONFIG['paths']['pretrained_smiles_encoder'])
for param in smiles_encoder.parameters():
    param.requires_grad = False # Freeze the encoder
smiles_encoder.to(device)
smiles_encoder.eval() # Set to evaluation mode

# Initialize Classifier
classifier_config = CONFIG['model_params']['classifier']
classifier = classifymodel(**classifier_config)
classifier.to(device)

print(f"SMILES Encoder parameters (frozen): {sum(p.numel() for p in smiles_encoder.parameters()):,}")
print(f"Classifier parameters (trainable): {sum(p.numel() for p in classifier.parameters() if p.requires_grad):,}")
print("✅ Models initialized and moved to device.")

---
### **Data Preparation & Core Logic** 🗂️
*Define the Training Blueprint*

With our models ready, this cell focuses on the data and the "how-to" of training. It will:

1.  **Load the training and validation datasets** specified in our configuration.
2.  **Create `DataLoaders`** to efficiently feed data to the GPU in batches.
3.  **Define the core `train_loop` and `validate_model` functions**, which contain the precise logic for one cycle of learning and evaluation.

Running this cell defines the blueprint for our training process. No training happens here yet, but it prepares all the necessary functions.

In [ ]:
# @title 2. Data Loading and Core Logic Definition
# @markdown This cell loads the training/validation data and defines the core `train` and `validate` functions.

# --- Helper Functions & Classes ---
def load_smiles_labels(smiles_path, labels_path):
    print(f"  > Loading SMILES: {os.path.basename(smiles_path)}")
    with open(smiles_path, 'r', encoding='utf-8') as f:
        smiles_list = [line.strip() for line in f if line.strip()]
    print(f"  > Loading Labels: {os.path.basename(labels_path)}")
    labels_tensor = torch.load(labels_path, map_location='cpu')
    return smiles_list, labels_tensor

class SmilesDataset(Dataset):
    def __init__(self, smiles_list, labels_tensor):
        self.smiles = smiles_list
        self.labels = labels_tensor
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.smiles[idx], self.labels[idx]

# --- Load data from paths defined in the config ---
print("Loading datasets...")
train_smiles, train_labels = load_smiles_labels(CONFIG['paths']['train_smiles'], CONFIG['paths']['train_labels'])
val_smiles, val_labels = load_smiles_labels(CONFIG['paths']['val_smiles'], CONFIG['paths']['val_labels'])

train_dataset = SmilesDataset(train_smiles, train_labels)
val_dataset = SmilesDataset(val_smiles, val_labels)

dl_params = CONFIG['dataloader_params']
pin_memory = (device.type == 'cuda')
train_loader = DataLoader(train_dataset, batch_size=dl_params['batch_size'], shuffle=True, num_workers=dl_params['num_workers'], pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=dl_params['batch_size'], shuffle=False, num_workers=dl_params['num_workers'], pin_memory=pin_memory)
print(f"✅ DataLoaders created. Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")


# --- Core Training & Validation Logic ---
def validate_model(smiles_encoder, classifier, val_loader, criterion, device):
    classifier.eval() # Only classifier needs to be set to eval mode
    running_loss = 0.0
    val_loader_tqdm = tqdm(val_loader, desc="Validating", unit="batch", leave=False)
    with torch.no_grad():
        for smiles_batch, labels_batch in val_loader_tqdm:
            labels = labels_batch.to(device)
            # Tokenize one-by-one to match original logic
            tokenizer = smiles_encoder.smiles_tokenizer
            encoded_smiles = [tokenizer.encode_plus(s, max_length=smiles_encoder.smiles_maxlen, padding='max_length', truncation=True, return_tensors='pt') for s in smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded_smiles], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded_smiles], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            with torch.amp.autocast(device_type=device.type):
                smiles_features = smiles_encoder.encode((input_ids, attention_mask), lengths)
                logits = classifier(smiles_features)
                loss = criterion(logits, labels)
            running_loss += loss.item() * labels.size(0)
            val_loader_tqdm.set_postfix(loss=loss.item())
    return running_loss / len(val_loader.dataset)

def train_loop(config, smiles_encoder, classifier, train_loader, val_loader, criterion, optimizer, device):
    scaler = GradScaler()
    # Unpack config params for clarity
    output_dir = config['paths']['output_dir']
    num_epochs = config['training_params']['num_epochs']
    num_best_models = config['model_save_params']['num_best_models']
    os.makedirs(output_dir, exist_ok=True)
    best_models_tracker, training_losses, validation_losses = [], [], []

    for epoch in range(num_epochs):
        classifier.train() # Set classifier to train mode
        running_loss = 0.0
        train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}", unit="batch")

        for smiles_batch, labels_batch in train_loader_tqdm:
            labels = labels_batch.to(device)
            # Tokenize one-by-one to match original logic
            tokenizer = smiles_encoder.smiles_tokenizer
            encoded_smiles = [tokenizer.encode_plus(s, max_length=smiles_encoder.smiles_maxlen, padding='max_length', truncation=True, return_tensors='pt') for s in smiles_batch]
            input_ids = torch.cat([item['input_ids'] for item in encoded_smiles], dim=0).to(device)
            attention_mask = torch.cat([item['attention_mask'] for item in encoded_smiles], dim=0).to(device)
            lengths = attention_mask.sum(dim=1)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type):
                with torch.no_grad():
                    smiles_features = smiles_encoder.encode((input_ids, attention_mask), lengths)
                logits = classifier(smiles_features)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            train_loader_tqdm.set_postfix(loss=loss.item())

        epoch_loss = running_loss / len(train_loader)
        training_losses.append(epoch_loss)
        val_loss = validate_model(smiles_encoder, classifier, val_loader, criterion, device)
        validation_losses.append(val_loss)
        print(f"Epoch {epoch + 1}/{num_epochs} -> Train Loss: {epoch_loss:.4f}, Val Loss: {val_loss:.4f}")

        # Save top N best models based on validation loss
        current_model_info = (val_loss, epoch + 1, None)
        if len(best_models_tracker) < num_best_models:
            best_models_tracker.append(current_model_info)
        elif val_loss < max(best_models_tracker, key=lambda x: x[0])[0]:
            worst_model_info = max(best_models_tracker, key=lambda x: x[0])
            if worst_model_info[2] and os.path.exists(worst_model_info[2]): os.remove(worst_model_info[2])
            best_models_tracker.remove(worst_model_info)
            best_models_tracker.append(current_model_info)

        for i, (loss, ep, _) in enumerate(best_models_tracker):
            if ep == epoch + 1:
                model_path = os.path.join(output_dir, f'classifier_epoch_{ep}_valloss_{loss:.4f}.pth')
                torch.save(classifier.state_dict(), model_path)
                best_models_tracker[i] = (loss, ep, model_path)
                print(f"  -> 💾 Checkpoint saved for epoch {ep} with Val Loss: {loss:.4f}")

    print('\n✅ Training complete.')
    print('🏆 Best validation models saved:')
    for loss, epoch, path, in sorted(best_models_tracker, key=lambda x: x[0]):
        print(f"  - Epoch {epoch}: Val Loss = {loss:.4f}, Path: {path}")

    return {'training_losses': training_losses, 'validation_losses': validation_losses}

print("✅ Core logic defined.")

---
### **Ignite the Engines!** 🚀
*Start the Training Process*

All systems are go! The models are initialized, the data is loaded, and the training functions are defined. This is the final step where you kick off the actual training process.

You can adjust the **`num_epochs`** to control how long the training runs. The script will provide real-time progress updates and automatically save the top-performing model checkpoints (based on the lowest validation loss) to your Google Drive.

**When you're ready, run this cell to begin training!**

In [ ]:
# @title 3. Start Training!
# @markdown Run this cell to start the training process for the SMILES Classifier.

# --- Setup Loss and Optimizer ---
criterion = torch.nn.CrossEntropyLoss().to(device)
opt_params = CONFIG['optimizer_params']
optimizer = AdamW(classifier.parameters(), lr=opt_params['learning_rate'], weight_decay=opt_params['weight_decay'])

print("--- Starting model training ---")

# --- Execute the Training Loop ---
history = train_loop(
    config=CONFIG,
    smiles_encoder=smiles_encoder,
    classifier=classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device
)

# --- Save Final Results ---
loss_save_path = os.path.join(CONFIG['paths']['output_dir'], CONFIG['model_save_params']['loss_log_file'])
with open(loss_save_path, 'w') as f:
    json.dump(history, f)
print(f"\nTraining loss history saved to: {loss_save_path}")

print("\n🎉🎉🎉 Training finished successfully! 🎉🎉🎉")